In [ ]:
!pip install langchain langchain-openai langchain-community python-dotenv -q

In [ ]:
import langchain
print(langchain.__version__)

In [ ]:
!pip install langchain-huggingface huggingface_hub -q

In [18]:
from google.colab import userdata
import os


os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HF_TOKEN1')

In [20]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    provider="auto",       # lets HF pick any provider that supports this model
    max_new_tokens=256,
    temperature=0.7,
)

chat_model = ChatHuggingFace(llm=llm_endpoint)

response = chat_model.invoke("Explain what LangChain is in 2 sentences.")
print(response.content)

LangChain is a framework that enables the creation of applications that connect language models with various tools and APIs, enhancing their utility and interoperability. It facilitates the chaining together of language models with different types of software and services to create more powerful and flexible AI systems.


In [21]:
from langchain_core.prompts import PromptTemplate

# Define a reusable template with placeholders
prompt_template = PromptTemplate(
    input_variables=["topic", "audience"],
    template="Explain {topic} to a {audience} in 3 clear sentences."
)

# Fill in the placeholders to generate the actual prompt
formatted_prompt = prompt_template.format(topic="vector embeddings", audience="beginner")
print("Formatted Prompt:\n", formatted_prompt)

# Pass it to the LLM
response = chat_model.invoke(formatted_prompt)
print("\nModel Response:\n", response.content)

Formatted Prompt:
 Explain vector embeddings to a beginner in 3 clear sentences.

Model Response:
 Vector embeddings are a way to represent text or numbers as vectors or arrays of numerical values. Each element in the vector can be thought of as a feature that helps describe the original data point. These vectors can then be used in various machine learning models to perform tasks like similarity searches or classification.


In [22]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define the prompt template
prompt = PromptTemplate(
    input_variables=["concept"],
    template="Explain the concept of {concept} in LangChain, with a simple analogy."
)

# Build the chain: prompt -> LLM -> output parser
chain = prompt | chat_model | StrOutputParser()

# Invoke the chain with input
result = chain.invoke({"concept": "chains"})
print(result)

Certainly! In the context of LangChain, "chains" refer to a sequence of steps or processes that are linked together to form a pipeline for generating responses based on prompts. Each step in the chain can be a different component or tool, such as a language model, a database query, or a function that processes the output of the previous step.

To make this concept more relatable, let's use a simple analogy:

Imagine you're baking a cake, and you have a recipe that involves several steps:

1. Preheat the oven.
2. Mix the dry ingredients.
3. Beat the eggs and sugar.
4. Combine the wet and dry ingredients.
5. Bake the cake.
6. Let it cool and decorate.

Each step is like a link in a chain, where the output of one step becomes the input for the next. Just like in LangChain, each step is crucial and contributes to the final product. If you don't preheat the oven (a step that can be considered an "agent" in LangChain), your cake won't bake properly. Similarly, if a step in a LangChain pipeli

In [23]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Store for conversation histories, keyed by session id
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Wrap the chat model with memory
chat_with_memory = RunnableWithMessageHistory(
    chat_model,
    get_session_history,
)

config = {"configurable": {"session_id": "user1"}}

# First message
r1 = chat_with_memory.invoke("My name is Aravind.", config=config)
print("Response 1:", r1.content)

# Second message - should remember the name
r2 = chat_with_memory.invoke("What's my name?", config=config)
print("Response 2:", r2.content)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Response 1: Hello Aravind! Nice to meet you. How can I assist you today? Feel free to tell me more about what you're interested in or need help with.
Response 2: Your name is Aravind.


In [24]:
from langchain_core.tools import tool
from langchain.agents import create_agent

# Define a simple custom tool
@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

# Create an agent that can use this tool
agent = create_agent(chat_model, tools=[get_word_length])

# Run the agent
result = agent.invoke({"messages": [{"role": "user", "content": "How many letters are in the word 'LangChain'?"}]})
print(result["messages"][-1].content)

The word "LangChain" contains 9 letters.


In [25]:
!pip install faiss-cpu langchain-huggingface sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 35.4 MB/s eta 0:00:00


In [26]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Simulate "loaded" documents (in real use, this comes from a DocumentLoader
# e.g. TextLoader, PyPDFLoader, WebBaseLoader)
docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="Agents in LangChain can decide which tools to call based on user input."),
    Document(page_content="Vector stores enable semantic search over unstructured text."),
]

# Create embeddings (converts text into numerical vectors)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Build a FAISS vector store from the documents
vectorstore = FAISS.from_documents(docs, embeddings)

# Perform a similarity search
query = "How do agents decide what to do?"
results = vectorstore.similarity_search(query, k=1)

print("Most relevant document:", results[0].page_content)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Most relevant document: Agents in LangChain can decide which tools to call based on user input.
